In [1]:
%load_ext autoreload
%autoreload 2

# LoRA-Laplace Training Pipeline

Full pipeline for the LoRA + diagonal Laplace Approximation uncertainty backend:

1. **Configuration** — paths, model, hyperparameters
2. **Fine-tune** — train LoRA adapters on (source, summary) pairs
3. **Fit Laplace approximation** — compute diagonal Fisher information and posterior variance
4. **Score calibration corpus** — collect sentence-level uncertainty scores
5. **Fit quantile normalizer** — map raw epistemic MI to a 0–100 display scale
6. **Upload to HuggingFace Hub** — push adapter, sampler and quantile config

All heavy-lifting is done by `src.lora_training` and `src.lora_laplace_backend` — this notebook only sets parameters and calls those modules.

**Input**: `data/summaries_v3.jsonl` (produced by `01_data_ingestion.ipynb`)

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = ROOT / "data"
MODELS_DIR = ROOT / "models"
CONFIG_DIR = ROOT / "config"
MODELS_DIR.mkdir(exist_ok=True)

SUMMARIES_FILE = DATA_DIR / "summaries_v3.jsonl"
ADAPTER_DIR    = MODELS_DIR / "bart-large-xsum-lora"
SCORES_FILE    = DATA_DIR / "uncertainty_scores_lora_laplace.jsonl"
SAMPLER_FILE   = ADAPTER_DIR / "laplace_sampler.npz"
QUANTILES_FILE = CONFIG_DIR / "uncertainty_quantiles_lora_laplace.json"

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "facebook/bart-large-xsum"

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_RANK           = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
LORA_LAYERS        = [10, 11]   # last 2 decoder layers (0-based, out of 12)

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS        = 10
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LEARNING_RATE = 3e-4
VAL_SPLIT     = 0.1
MAX_SRC_LEN   = 512
MAX_TGT_LEN   = 128
SEED          = 42
BF16          = True   # set False if bf16 is not supported on your device

# ── Laplace calibration ───────────────────────────────────────────────────────
CALIBRATION_SPLIT = 0.1
PRIOR_PRECISION   = 1.0
SAMPLE_COUNT      = 10

# ── HuggingFace Hub ───────────────────────────────────────────────────────────
HF_REPO_ID   = "rdisipio/summarizer-uncertainty-models"
HF_SUBFOLDER = "bart-large-xsum-lora-laplace"

print("Config OK")

Config OK


## 2. Load and split data

In [3]:
from src.lora_training import load_pairs, split_pairs

pairs = load_pairs(str(SUMMARIES_FILE))
train_pairs, val_pairs = split_pairs(pairs, val_split=VAL_SPLIT, seed=SEED)
print(f"Train: {len(train_pairs)}  |  Val: {len(val_pairs)}")

Train: 1707  |  Val: 189


## 3. Fine-tune with LoRA

In [4]:
import numpy as np, torch
from src.lora_training import build_lora_model, train_lora

torch.manual_seed(SEED)
np.random.seed(SEED)

model, tokenizer = build_lora_model(
    model_name=BASE_MODEL,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    layers_to_transform=LORA_LAYERS,
    layers_pattern="decoder",
)

train_lora(
    model=model,
    tokenizer=tokenizer,
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    output_dir=str(ADAPTER_DIR),
    max_source_length=MAX_SRC_LEN,
    max_target_length=MAX_TGT_LEN,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    bf16=BF16,
    seed=SEED,
)

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 131,072 || all params: 509,362,176 || trainable%: 0.0257


Tokenising train set:   0%|          | 0/1707 [00:00<?, ? examples/s]

Tokenising validation set:   0%|          | 0/189 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/disipio/.local/share/virtualenvs/summarizer-uncertainty-ml-lNq8rFAG/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,8.965371,2.172894
2,8.729723,2.058517
3,8.259635,2.021496
4,8.198940,1.990814
5,8.229981,1.974862
6,8.108324,1.976778
7,7.772668,1.966401
8,8.181145,1.965391
9,7.997610,1.957247
10,7.938846,1.956882


/Users/disipio/.local/share/virtualenvs/summarizer-uncertainty-ml-lNq8rFAG/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/disipio/.local/share/virtualenvs/summarizer-uncertainty-ml-lNq8rFAG/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/disipio/.local/share/virtualenvs/summarizer-uncertainty-ml-lNq8rFAG/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/disipio/.local/share/virtualenvs/summarizer-uncertainty-ml-lNq8rFAG/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning

## 4. Fit the diagonal Laplace approximation

In [5]:
import random
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from src.lora_laplace_backend import LoraLaplaceBackend, fit_laplace_approximation, save_laplace_sampler
from src.lora_training import load_pairs

# Re-load all pairs and carve out the calibration subset
all_pairs = [(r["source"], r["target"]) for r in load_pairs(str(SUMMARIES_FILE))]
random.seed(SEED)
random.shuffle(all_pairs)
cal_size = max(1, int(len(all_pairs) * CALIBRATION_SPLIT))
calibration_data = all_pairs[:cal_size]
print(f"Calibration samples: {len(calibration_data)}")

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
peft_model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=True)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

backend = LoraLaplaceBackend(peft_model=peft_model, tokenizer=tok, device=device)

print("Fitting Laplace approximation...")
sampler = fit_laplace_approximation(
    backend=backend,
    calibration_data=calibration_data,
    prior_precision=PRIOR_PRECISION,
)

save_laplace_sampler(sampler, str(SAMPLER_FILE))
print(f"Sampler saved to {SAMPLER_FILE}")

Calibration samples: 189
Device: mps


Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Fitting Laplace approximation...
Sampler saved to /Users/disipio/development/summarizer-uncertainty-ml/models/bart-large-xsum-lora/laplace_sampler.npz


## 5. Score the corpus and collect uncertainty values

In [6]:
# ── Reload backend and sampler from disk (skip if section 4 was just run) ────
# Run this cell if the kernel was restarted and backend/sampler are not in memory.
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from src.lora_laplace_backend import LoraLaplaceBackend, load_laplace_sampler
from src.lora_training import load_pairs
import random

if "backend" not in dir() or "sampler" not in dir():
    device = (
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
    print(f"Reloading backend from {ADAPTER_DIR} on {device}...")
    base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
    peft_model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=True)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    backend = LoraLaplaceBackend(peft_model=peft_model, tokenizer=tok, device=device)
    print(f"Reloading sampler from {SAMPLER_FILE}...")
    sampler = load_laplace_sampler(str(SAMPLER_FILE))

# Rebuild scoring_pairs and all_pairs regardless (cheap — just reads JSONL)
all_pairs = [(r["source"], r["target"]) for r in load_pairs(str(SUMMARIES_FILE))]
random.seed(SEED)
random.shuffle(all_pairs)
cal_size = max(1, int(len(all_pairs) * CALIBRATION_SPLIT))
scoring_pairs = all_pairs[cal_size:]
print(f"Ready. {len(scoring_pairs)} pairs to score. Base model: {BASE_MODEL}, LoRA adapter: {ADAPTER_DIR.name}")

Ready. 1707 pairs to score. Base model: facebook/bart-large-xsum, LoRA adapter: bart-large-xsum-lora


In [7]:
from tqdm.auto import tqdm
from src.scorer import SummaryUncertaintyScorer
from src.data_pipeline import write_jsonl

scorer = SummaryUncertaintyScorer(backend=backend, posterior_sampler=sampler)
scoring_pairs = all_pairs[cal_size:]

all_sentence_scores = []
skipped = 0

SCORES_FILE.unlink(missing_ok=True)
for source, summary in tqdm(scoring_pairs, desc="Scoring"):
    try:
        result = scorer.score_summary(source=source, summary=summary,
                                      sample_count=5, seed=SEED)
    except Exception as e:
        print(f"[ERROR] {e}")
        skipped += 1
        continue

    sentence_scores = [
        {"sentence_index": s.sentence_index, "sentence_text": s.sentence_text,
         "uncertainty": s.uncertainty}
        for s in result.sentence_results
    ]
    all_sentence_scores.extend(sentence_scores)
    if sentence_scores:
        avg = sum(s["uncertainty"] for s in sentence_scores) / len(sentence_scores)
        write_jsonl(str(SCORES_FILE), {"sentence_scores": sentence_scores, "uncertainty_avg": avg})

print(f"Scored {len(scoring_pairs) - skipped} summaries, {len(all_sentence_scores)} sentences. Skipped: {skipped}")

Scoring:   0%|          | 0/1707 [00:00<?, ?it/s]

Scored 1707 summaries, 4595 sentences. Skipped: 0


## 6. Fit quantile normalizer

In [12]:
import json

scores = sorted(s["uncertainty"] for s in all_sentence_scores if s.get("uncertainty") is not None)
n = len(scores)
boundaries = [scores[0], scores[n//4], scores[n//2], scores[3*n//4], scores[-1]]

QUANTILES_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(QUANTILES_FILE, "w") as f:
    json.dump({"boundaries": boundaries}, f, indent=2)
    f.write("\n")

print(f"Sentences: {n}  |  Boundaries: {[f'{b:.4f}' for b in boundaries]}")


Sentences: 4595  |  Boundaries: ['0.0034', '0.1691', '0.2281', '0.2929', '0.5834']


## 7. Upload to HuggingFace Hub

In [13]:
from src.lora_training import upload_to_hub

upload_to_hub(
    files=[
        (str(ADAPTER_DIR / "adapter_config.json"),       f"{HF_SUBFOLDER}/adapter_config.json"),
        (str(ADAPTER_DIR / "adapter_model.safetensors"), f"{HF_SUBFOLDER}/adapter_model.safetensors"),
        (str(ADAPTER_DIR / "tokenizer.json"),            f"{HF_SUBFOLDER}/tokenizer.json"),
        (str(ADAPTER_DIR / "tokenizer_config.json"),     f"{HF_SUBFOLDER}/tokenizer_config.json"),
        (str(SAMPLER_FILE),                              f"{HF_SUBFOLDER}/laplace_sampler.npz"),
        (str(QUANTILES_FILE),                            f"{HF_SUBFOLDER}/uncertainty_quantiles_lora_laplace.json"),
    ],
    repo_id=HF_REPO_ID,
)

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


In [10]:
from src.scorer import SummaryUncertaintyScorer

scorer = SummaryUncertaintyScorer(backend=backend, posterior_sampler=sampler)
source, summary = all_pairs[0]

In [11]:
for k in [None, 5]:
    result = scorer.score_summary(source=source, summary=summary, sample_count=100, seed=SEED, top_k_tokens=k)
    print(f"\n--- top_k_tokens={k} ---")
    for s in result.sentence_results:
        print(f"  [{s.uncertainty:.4f}] {s.sentence_text[:80]}")



--- top_k_tokens=None ---
  [0.5621] Climate change is causing some serious long-term issues for our oceans, like ris
  [0.5654] It's also pushing wildlife to move to cooler areas, like birds shifting north, b
  [0.6458] Overall, the future of our climate and its impact on nature is pretty uncertain 

--- top_k_tokens=5 ---
  [0.7737] Climate change is causing some serious long-term issues for our oceans, like ris
  [0.7657] It's also pushing wildlife to move to cooler areas, like birds shifting north, b
  [0.8035] Overall, the future of our climate and its impact on nature is pretty uncertain 
